In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score


plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


housing = fetch_california_housing(as_frame=True)
df = housing.frame


print("--- Dataset Shape ---")
print(f"Rows: {df.shape[0]}, Features: {df.shape[1] - 1}, Target: 1\n")

print("--- Feature Information ---")
df.info()

print("\n--- Statistical Summary ---")
display(df.describe().round(3))


X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

In [ ]:

def evaluate_regression(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    return {
        'Model': model_name,
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4),
        'R2': round(r2, 4)
    }


models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01) # Small alpha to prevent severe underfitting
}

results = []
for name, estimator in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', estimator)
    ])
    results.append(evaluate_regression(pipe, X_train, y_train, X_test, y_test, name))


poly_features = ['MedInc', 'HouseAge']
other_features = [col for col in X.columns if col not in poly_features]


preprocessor = ColumnTransformer(
    transformers=[
        ('poly', PolynomialFeatures(degree=2, include_bias=False), poly_features),
        ('pass', 'passthrough', other_features)
    ]
)

poly_pipe = Pipeline([
    ('prep', preprocessor),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

results.append(evaluate_regression(poly_pipe, X_train, y_train, X_test, y_test, 'Ridge + Poly Features (MedInc, HouseAge)'))


reg_df = pd.DataFrame(results)
display(reg_df)


ridge_model = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
ridge_model.fit(X_train, y_train)
feature_weights = pd.Series(
    ridge_model.named_steps['model'].coef_, 
    index=X.columns
).sort_values(key=abs, ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=feature_weights.values, y=feature_weights.index, palette='viridis')
plt.title('Feature Importance (Ridge Coefficients - Standardized)')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.show()

In [ ]:

median_price = y_train.median()
y_train_bin = (y_train > median_price).astype(int)
y_test_bin = (y_test > median_price).astype(int)


binary_models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42)
}

bin_results = []
for name, model in binary_models.items():
    model.fit(X_train, y_train_bin)
    y_pred = model.predict(X_test)
    
    bin_results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test_bin, y_pred), 4),
        'Precision': round(precision_score(y_test_bin, y_pred), 4),
        'Recall': round(recall_score(y_test_bin, y_pred), 4),
        'F1-Score': round(f1_score(y_test_bin, y_pred), 4)
    })
    

    cm = confusion_matrix(y_test_bin, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

display(pd.DataFrame(bin_results))


quartiles = pd.qcut(y_train, q=4, labels=[0, 1, 2, 3], retbins=True)[1]
y_train_multi = pd.cut(y_train, bins=quartiles, labels=[0, 1, 2, 3], include_lowest=True).astype(int)
y_test_multi = pd.cut(y_test, bins=quartiles, labels=[0, 1, 2, 3], include_lowest=True).astype(int)


rf_param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [10, 15],
    'min_samples_split': [5, 10]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train_multi)

best_rf = rf_grid.best_estimator_
y_pred_multi = best_rf.predict(X_test)

print(f"Best Random Forest Params: {rf_grid.best_params_}")
print(f"Multi-Class Accuracy: {accuracy_score(y_test_multi, y_pred_multi):.4f}\n")
print("--- Classification Report (Multi-Class) ---")
print(classification_report(y_test_multi, y_pred_multi, target_names=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']))

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained Variance Ratio by 2 Components: {pca.explained_variance_ratio_.sum():.4f}")


kmeans_metrics = []
for k in [3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    
    inertia = km.inertia_
    sil_score = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    
    kmeans_metrics.append({'k': k, 'Inertia (SSE)': round(inertia, 2), 'Silhouette Score': round(sil_score, 4)})

display(pd.DataFrame(kmeans_metrics))


kmeans_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans_4.fit_predict(X_scaled)


sample_indices = np.random.choice(X_scaled.shape[0], size=5000, replace=False)
hierarchical = AgglomerativeClustering(n_clusters=4, linkage='ward')
hier_labels = hierarchical.fit_predict(X_scaled[sample_indices])


fig, axes = plt.subplots(1, 2, figsize=(16, 6))


sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=df['KMeans_Cluster'],
    palette='Set1', s=15, alpha=0.6, ax=axes[0]
)
axes[0].set_title('KMeans Clusters (k=4) in PCA 2D Space')
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')


geo_scatter = sns.scatterplot(
    data=df, x='Longitude', y='Latitude',
    hue='KMeans_Cluster', palette='Set1',
    size='MedHouseVal', sizes=(10, 100),
    alpha=0.6, ax=axes[1]
)
axes[1].set_title('Geographic Distribution of Clusters')
plt.tight_layout()
plt.show()